# Practice Lab H: Naive Bayes Spam Filter (emails.csv)

**Area:** probability · **Related lessons/concepts:** L50; Quiz 13

**You will be able to:**


In [1]:
import numpy as np
import sympy as sp
import matplotlib
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

## 1. Load the course data (fallback: tiny synthetic corpus)

In [2]:
import os,re,pandas as pd
RAW="data/"  # optional course data; synthetic stand-in is used when absent
p=RAW+"C3/w1/C3w1_graded_lab/emails.csv"
if os.path.exists(p): df=pd.read_csv(p)
else:
    df=pd.DataFrame({"text":["win money now","free lottery win","meeting at noon","project notes attached"]*50,"spam":[1,1,0,0]*50})
print(df.shape, df.spam.mean()); print(df.text.iloc[0][:120])

(5728, 2) 0.2388268156424581
Subject: naturally irresistible your corporate identity  lt is really hard to recollect a company : the  market is full 


## 2. Train/test split and word counts

In [3]:
from collections import Counter
rng=np.random.default_rng(0); idx=rng.permutation(len(df)); cut=int(0.8*len(df)); tr,te=df.iloc[idx[:cut]],df.iloc[idx[cut:]]
tok=lambda s:set(re.findall(r"[a-z]+",s.lower()))     # presence of a word (binary features)
cnt={c:Counter() for c in (0,1)}; ndoc={c:0 for c in (0,1)}
for t,l in zip(tr.text,tr.spam):
    ndoc[int(l)]+=1; cnt[int(l)].update(tok(t))
vocab={w for c in cnt for w in cnt[c] if sum(cnt[k][w] for k in cnt)>=5}
print(len(vocab),"words;",ndoc)

8609 words; {0: 3515, 1: 1067}


## 3. Classifier (Bernoulli Naive Bayes with Laplace smoothing)

In [4]:
V=sorted(vocab); prior={c:ndoc[c]/sum(ndoc.values()) for c in (0,1)}
pw={c:np.array([(cnt[c][w]+1)/(ndoc[c]+2) for w in V]) for c in (0,1)}
vi={w:i for i,w in enumerate(V)}
def feats(t):
    v=np.zeros(len(V)); 
    for w in tok(t):
        if w in vi: v[vi[w]]=1
    return v
def logp(v,c): return np.log(prior[c])+(v*np.log(pw[c])+(1-v)*np.log(1-pw[c])).sum()
pred=np.array([int(logp(feats(t),1)>logp(feats(t),0)) for t in te.text]); y=te.spam.values.astype(int)
tp=((pred==1)&(y==1)).sum(); fp=((pred==1)&(y==0)).sum(); fn=((pred==0)&(y==1)).sum()
print("accuracy",(pred==y).mean(),"precision",tp/max(tp+fp,1),"recall",tp/max(tp+fn,1))

accuracy 0.9650959860383944 precision 0.887240356083086 recall 0.9933554817275747


## 4. Most spam-indicative words

In [5]:
ratio=np.log(pw[1]/pw[0]); top=np.argsort(ratio)[::-1][:10]; print([V[i] for i in top])

['viagra', 'stationery', 'sex', 'andmanyother', 'oniine', 'wili', 'professionai', 'breath', 'projecthoneypot', 'wiil']


## Exercises

**Exercise 1.** Compare with scikit-learn's BernoulliNB on the same features.

In [6]:
# your code here

**Exercise 2.** Change the smoothing constant from 1 to 0.01 and 10: what happens to accuracy?

In [7]:
# your code here

## Solutions
*(Try the exercises first!)*

In [8]:
# Solution 1
from sklearn.naive_bayes import BernoulliNB
Xtr=np.array([feats(t) for t in tr.text]); Xte=np.array([feats(t) for t in te.text]); sk=BernoulliNB(alpha=1).fit(Xtr,tr.spam.astype(int)); print((sk.predict(Xte)==y).mean()); assert (sk.predict(Xte)==y).mean()>0.85

0.9650959860383944


In [9]:
# Solution 2
def acc_alpha(al):
    pw_={c:np.array([(cnt[c][w]+al)/(ndoc[c]+2*al) for w in V]) for c in (0,1)}
    lp=lambda v,c:np.log(prior[c])+(v*np.log(pw_[c])+(1-v)*np.log(1-pw_[c])).sum()
    pr=np.array([int(lp(feats(t),1)>lp(feats(t),0)) for t in te.text]); return (pr==y).mean()
res={al:round(acc_alpha(al),4) for al in (0.01,1,10)}; print(res); assert all(v>0.8 for v in res.values())

{0.01: np.float64(0.9729), 1: np.float64(0.9651), 10: np.float64(0.8813)}
